# **3. Обучение бейзлайна**

**Цель:** получить первую рабочую модель и метрику Precision@Recall>=0.7.

Для бейзлайна выберем три модели и сравним их качество: 

- логистическая регрессия

- случайный лес

- градиентный бустинг

Для валидации будем использовать `Stratified KFold` - поскольку у нас сильный дисбаланс классов (примерно 1 к 11).

In [1]:
import sys
import os

# необходимо для того чтобы ноутбук увидел src
current_dir = os.getcwd()

if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings

from src.data_loader import load_train, load_events
from src.features_baseline import build_features
from src.metrics import precision_at_recall

warnings.filterwarnings('ignore')

N_SPLITS = 5
RANDOM_STATE = 42

Загрузка данных:

In [3]:
train, events = load_train(), load_events()
features = build_features(events, train, verbose=True)
features.head()

Событий в окне: 198504
Итого признаков: 46


,cookie_id,event_count,unique_items,unique_categories,unique_locations,unique_event_types,ua_length,has_headless,has_bot_pattern,n_unique_ua,...,ratio_item_view,ratio_login,ratio_photo_swipe,ratio_search_results_view,ratio_seller_page_view,has_favorite_add,has_login,has_captcha,item_view_to_photo_ratio,target
0,ck_000c95f1408dcb00,16,9,2,8,4,117.0,0,0,1,...,0.687500,0.000000,0.062500,0.187500,0.062500,0,0,0,5.500000,0
1,ck_000e8c52636e3bec,34,20,4,12,7,80.0,0,0,1,...,0.323529,0.029412,0.147059,0.264706,0.088235,1,1,0,1.833333,0
2,ck_0010e31baa4a1fb7,16,8,3,4,6,130.0,0,0,1,...,0.437500,0.062500,0.062500,0.250000,0.062500,1,1,0,3.500000,0
3,ck_0010ec3874fb5378,74,31,2,19,6,130.0,0,0,1,...,0.337838,0.013514,0.189189,0.270270,0.054054,1,1,0,1.666667,0
4,ck_001722063b94cae0,15,8,3,9,4,123.0,0,0,1,...,0.533333,0.000000,0.066667,0.333333,0.000000,0,0,0,4.000000,0


In [4]:
X = features.drop(columns=['cookie_id', 'target'])
y = features['target']

Общие функции для валидации:

In [5]:
def evaluate_cv(model, X: pd.DataFrame, y: pd.Series, n_splits: int = N_SPLITS):
    """OOF-оценка модели. Метрика считается один раз на всех OOF-score."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    oof = np.zeros(len(X))
    fold_iters = list(skf.split(X, y))

    for fold, (tr_idx, va_idx) in enumerate(fold_iters, 1):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
        print(f"  fold {fold}/{n_splits} - обучено")

    y_arr = y.values
    p_at_r = precision_at_recall(y_arr, oof)
    pr_auc = average_precision_score(y_arr, oof)
    roc_auc = roc_auc_score(y_arr, oof)

    return {
        "P@R>=0.7": p_at_r,
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "oof": oof,
    }


def constant_baseline(y: pd.Series) -> float:
    score = np.full(len(y), 1.0)
    return precision_at_recall(y.values, score)

### **3.1. Обучение логистической регрессии**

In [6]:
logreg = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            penalty='elasticnet',
            solver='saga',
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

# Сравнение с константным baseline
const_score = constant_baseline(y)
print(f"\nConstant baseline P@R>=0.7: {const_score:.4f}")

# Кросс-валидация
print(f"\nStratifiedKFold({N_SPLITS}) - LogisticRegression:")
lr_res = evaluate_cv(logreg, X, y, n_splits=N_SPLITS)

print("\nРезультат:\n")
print(f"  Precision@Recall>=0.70 : {lr_res['P@R>=0.7']:.4f}")
print(f"  PR-AUC                 : {lr_res['PR-AUC']:.4f}")
print(f"  ROC-AUC                : {lr_res['ROC-AUC']:.4f}")


Constant baseline P@R>=0.7: 0.0811

StratifiedKFold(5) - LogisticRegression:
  fold 1/5 - обучено
  fold 2/5 - обучено
  fold 3/5 - обучено
  fold 4/5 - обучено
  fold 5/5 - обучено

Результат:

  Precision@Recall>=0.70 : 0.3127
  PR-AUC                 : 0.5341
  ROC-AUC                : 0.8636


### **3.2. Обучение случайного леса**

In [7]:
random_forest = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ])

const_score = constant_baseline(y)
print(f"\nConstant baseline P@R>=0.7: {const_score:.4f}")

print(f"\nStratifiedKFold({N_SPLITS}) - RandomForestClassifier:\n")
rf_res = evaluate_cv(random_forest, X, y, n_splits=N_SPLITS)

print("\nРезультат:\n")
print(f"  Precision@Recall>=0.70 : {rf_res['P@R>=0.7']:.4f}")
print(f"  PR-AUC                 : {rf_res['PR-AUC']:.4f}")
print(f"  ROC-AUC                : {rf_res['ROC-AUC']:.4f}")      


Constant baseline P@R>=0.7: 0.0811

StratifiedKFold(5) - RandomForestClassifier:

  fold 1/5 - обучено
  fold 2/5 - обучено
  fold 3/5 - обучено
  fold 4/5 - обучено
  fold 5/5 - обучено

Результат:

  Precision@Recall>=0.70 : 0.4716
  PR-AUC                 : 0.6790
  ROC-AUC                : 0.8936


### **3.3. Обучение градиентного бустинга**

In [8]:
cat_boost = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            iterations=700,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3.0,
            loss_function="Logloss",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )),
    ])

const_score = constant_baseline(y)
print(f"\nConstant baseline P@R>=0.7: {const_score:.4f}")

print(f"\nStratifiedKFold({N_SPLITS}) - CatBoostClassifier:\n")
ct_res = evaluate_cv(cat_boost, X, y, n_splits=N_SPLITS)

print("\nРезультат:\n")
print(f"  Precision@Recall>=0.70 : {ct_res['P@R>=0.7']:.4f}")
print(f"  PR-AUC                 : {ct_res['PR-AUC']:.4f}")
print(f"  ROC-AUC                : {ct_res['ROC-AUC']:.4f}")  


Constant baseline P@R>=0.7: 0.0811

StratifiedKFold(5) - CatBoostClassifier:

  fold 1/5 - обучено
  fold 2/5 - обучено
  fold 3/5 - обучено
  fold 4/5 - обучено
  fold 5/5 - обучено

Результат:

  Precision@Recall>=0.70 : 0.5464
  PR-AUC                 : 0.7107
  ROC-AUC                : 0.9017


### **3.4. Выводы**

Соберем все результаты в единую таблицу:

In [9]:
results = [
    {"Model": "LogReg",  **{k: lr_res[k] for k in ("P@R>=0.7", "PR-AUC", "ROC-AUC")}},
    {"Model": "RandomForest",  **{k: rf_res[k] for k in ("P@R>=0.7", "PR-AUC", "ROC-AUC")}},
    {"Model": "CatBoost",      **{k: ct_res[k] for k in ("P@R>=0.7", "PR-AUC", "ROC-AUC")}},
]

df = pd.DataFrame(results).rename(columns={
    "P@R>=0.7": "P@R>=0.70",
    "PR-AUC":   "PR-AUC",
    "ROC-AUC":  "ROC-AUC",
})
df = df.sort_values("P@R>=0.70", ascending=False).reset_index(drop=True)

df.loc[len(df)] = ["Constant baseline", const_score, np.nan, np.nan]

print("=" * 60)
print(f"{'Model':<20} {'P@R>=0.70':>10} {'PR-AUC':>10} {'ROC-AUC':>10}")
print("-" * 60)
for _, row in df.iterrows():
    pr  = "-" if pd.isna(row["PR-AUC"])  else f"{row['PR-AUC']:.4f}"
    roc = "-" if pd.isna(row["ROC-AUC"]) else f"{row['ROC-AUC']:.4f}"
    print(f"{row['Model']:<20} {row['P@R>=0.70']:>10.4f} {pr:>10} {roc:>10}")
print("=" * 60)

Model                 P@R>=0.70     PR-AUC    ROC-AUC
------------------------------------------------------------
CatBoost                 0.5464     0.7107     0.9017
RandomForest             0.4716     0.6790     0.8936
LogReg                   0.3127     0.5341     0.8636
Constant baseline        0.0811          -          -


**Ключевые наблюдения:**

1. **CatBoost — явный лидер** В целом как и ожидалось, градиентный бустинг лучше всех показывает себя при работе с табличными данными. Поэтому в финальной модели выбор будем отдавать именно CatBoost.

2. **Большой разрыв между ROC-AUC и целевой метрикой** Это означает, что модель неплохо ранжирует куки, но при жёстком требовании `Recall ≥ 0.70` precision сильно проседает. То есть уверенности в верхней части распределения у модели - нет.

3. **Линейная модель сильно отстаёт**. Поведенческие паттерны ботов
 нелинейны и линейный классификатор их не ловит.

4. **Constant baseline = 0.0811**. Фактическая доля ботов в выборке. Наша лучшая модель поднимает precision с 8% до ~55% при сохранении 70% полноты - это существенный прирост, но потолок ещё не достигнут.

**Вывод**: Baseline-модели неплохо себя показали, но уперлись в потолок текущего набора фич. На данный момент основаная работа будет состоять не в тюнинге гиперпарамтов, а в features engineering.

Также, перед генерацией новых признаков, есть смысл посмотреть на важность фич в бейзлайне:

In [10]:
final_cb = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            iterations=700,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3.0,
            loss_function="Logloss",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )),
    ])

# Обучаем CatBoost на всём train - только для importance
final_cb.fit(X, y)

cat_clf = final_cb.named_steps["clf"]
importance = pd.Series(
    cat_clf.get_feature_importance(),
    index=X.columns,
).sort_values(ascending=False)

TOP_N = 20
top = importance.head(TOP_N)

print(f"Топ-{TOP_N} признаков (CatBoost feature importance):")
for name, val in top.items():
    print(f"  {name:<38} {val:>6.2f}")

print(f"\nВсего признаков: {len(importance)}")
print(f"Вносят >1%:       {(importance > 1.0).sum()}")
print(f"Вносят >0.5%:     {(importance > 0.5).sum()}")
print(f"Вносят <0.1%:     {(importance < 0.1).sum()} (кандидаты на выброс)")

Топ-20 признаков (CatBoost feature importance):
  median_interval                         13.15
  unique_locations                         5.80
  unique_categories                        5.41
  mean_search_page                         5.09
  min_interval                             5.04
  cookie_age_log                           4.13
  pointer_x_range                          3.97
  mean_interval                            3.76
  ratio_favorite_add                       3.41
  ratio_photo_swipe                        3.21
  event_count                              3.01
  seller_private_ratio                     2.88
  ua_length                                2.86
  ratio_item_view                          2.85
  seller_pro_ratio                         2.80
  ratio_contact_phone_show                 2.54
  unique_items                             2.51
  hour_std                                 2.43
  ratio_seller_page_view                   2.16
  item_view_to_photo_ratio              

In [11]:
low_imp = importance[importance < 0.1].sort_values()

if len(low_imp) > 0:
    print(f"\nКандидаты на выброс (importance < 0.1%):")
    for name, val in low_imp.items():
        print(f"  {name:<38} {val:>6.3f}")
    print(f"\nИтого: {len(low_imp)} фич из {len(importance)} "
          f"({100 * len(low_imp) / len(importance):.1f}%)")
else:
    print("\nНет фич с importance < 0.1% — чистить нечего.")


Кандидаты на выброс (importance < 0.1%):
  is_created_in_window                    0.000
  has_bot_pattern                         0.023
  has_captcha                             0.033
  has_headless                            0.036
  ratio_captcha_shown                     0.037
  has_any_pointer                         0.079

Итого: 6 фич из 46 (13.0%)


Основные выводы:

1. **Временные признаки - главный сигнал для модели**, можно подумать об их усилении.

2. **Разнообразие категорий, локаций и товаров** - также влияет на предикт модели, но меньше, чем ожидалось. Необходимо добавить нормированные версии: items_per_event, categories_per_event - сейчас абсолютные счётчики смешаны с длиной сессии.

3. **Флаги "headless / bot pattern / captcha" - практически нулевые**. Это значит, что либо боты хорошо маскируют UA, либо эти события слишком редкие, чтобы быть полезными. Нужны более тонкие UA-паттерны или вообще убрать эти фичи.

4. **Поисковое поведение.**  `mean_search_page` (5.09%) и `max_search_page` (1.89%) - боты листают выдачу гораздо глубже человека.

5. **Поведенческие маркеры работают, но умеренно.**  `ratio_favorite_add`, `ratio_photo_swipe`, `ratio_contact_*` - человек взаимодействует с контентом, бот забирает данные.